# Interactive Agent Session: Chapter3_OpenAI_DataScientist

This Jupyter notebook provides a sandbox to run AI Agent code securely block-by-block. Make sure you have your `.env` configured properly before executing the agent loop below.

In [ ]:
import openai

client = openai.Client(api_key="YOUR_OPENAI_API_KEY")

# 1. Upload the business file to OpenAI
file = client.files.create(
  file=open("q1_sales_data.csv", "rb"),
  purpose='assistants'
)

# 2. Create the Assistant with Code execution enabled
assistant = client.beta.assistants.create(
  name="Data Scientist",
  instructions="Analyze CSVs, write and run Python code natively to identify trends, and generate matplotlib charts.",
  tools=[{"type": "code_interpreter"}], 
  model="gpt-4-turbo"
)

# 3. Create a Thread (Stateful Memory managed by OpenAI)
thread = client.beta.threads.create()
client.beta.threads.messages.create(
  thread_id=thread.id, role="user",
  content="Analyze this CSV. Tell me the top 3 regions and generate a revenue chart.",
  attachments=[{ "file_id": file.id, "tools": [{"type": "code_interpreter"}] }]
)

# 4. Trigger Execution
run = client.beta.threads.runs.create_and_poll(thread_id=thread.id, assistant_id=assistant.id)

# 5. Retrieve Results
if run.status == 'completed': 
    messages = client.beta.threads.messages.list(thread_id=thread.id)
    for msg in messages.data:
        if msg.content[0].type == 'text':
            print("Assistant Analysis:\n", msg.content[0].text.value)
        if msg.content[0].type == 'image_file':
            image_file_id = msg.content[0].image_file.file_id
            print(f"Chart generated! File ID: {image_file_id}")
